# Pipeline de tri de pages financières (PDF normaux + scannés)

Ce notebook orchestre un pipeline **modulaire** composé de fichiers Python séparés :

| Fichier | Rôle |
|---|---|
| `config.py` | Mots-clés EN/FR, poids, seuils — **modifiez les règles ici** |
| `text_extraction.py` | Extraction de texte (natif / OCR / hybride) — **remplacez l'OCR ici** |
| `relevance.py` | Scoring de pertinence (mots-clés, densité numérique, zero-shot optionnel) |
| `pdf_processor.py` | Traite un PDF ou un dossier entier, écrit les PDF filtrés |
| `report.py` | Génère le fichier Excel global |

**Pourquoi c'est modulaire ?** Chaque brique dépend d'une interface abstraite
(`TextExtractor`, `RelevanceScorer`) et non d'une implémentation précise.
Par exemple, si demain vous voulez remplacer Tesseract par EasyOCR ou une API
cloud, il suffit d'écrire une nouvelle classe héritant de `TextExtractor` dans
`text_extraction.py` — **aucune autre partie du pipeline n'a besoin de changer**.


## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # s'assure que les modules du dossier sont importables

from text_extraction import NativeTextExtractor, TesseractOCRExtractor, HybridTextExtractor
from relevance import build_default_composite_scorer, KeywordScorer, NumericDensityScorer, CompositeScorer
from pdf_processor import process_folder
from report import write_excel_report
import config


## 2. Paramètres

- `INPUT_DIR` : dossier contenant les PDF financiers à traiter (normaux ou scannés)
- `OUTPUT_DIR` : dossier où seront écrits les PDF filtrés (mêmes noms de fichiers, mais seulement les pages pertinentes)
- `EXCEL_REPORT_PATH` : chemin du fichier Excel récapitulatif global (situé **en dehors** de `OUTPUT_DIR`)


In [ ]:
INPUT_DIR = "input_pdfs"                 # <-- dossier avec vos PDF financiers
OUTPUT_DIR = "output_pdfs"               # <-- dossier de sortie (PDF filtrés)
EXCEL_REPORT_PATH = "rapport_global.xlsx"  # <-- fichier Excel global (hors OUTPUT_DIR)

os.makedirs(INPUT_DIR, exist_ok=True)
print(f"Déposez vos PDF dans le dossier: {os.path.abspath(INPUT_DIR)}")


## 3. Configuration des règles de détection (optionnel)

Toutes les valeurs ci-dessous viennent de `config.py`. Vous pouvez soit éditer
directement `config.py`, soit surcharger certains paramètres ici sans toucher
au fichier (utile pour tester rapidement différents seuils).

In [ ]:
settings = dict(config.DEFAULT_SETTINGS)  # copie modifiable

# Exemples de réglages que vous pouvez ajuster :
# settings["relevance_threshold"] = 0.30       # seuil plus permissif
# settings["numeric_density_threshold"] = 0.05
# settings["use_zero_shot"] = True             # nécessite: pip install transformers torch
# settings["native_text_min_chars"] = 30

settings


## 4. Construction des modules (extracteur de texte + scorer de pertinence)

C'est ICI que vous changeriez de moteur OCR si besoin : remplacez
`TesseractOCRExtractor` par votre propre classe (héritant de `TextExtractor`)
dans `text_extraction.py`, puis substituez-la ci-dessous. Le reste du notebook
ne change pas.

In [ ]:
# --- Extraction de texte ---
ocr_extractor = TesseractOCRExtractor(
    lang=settings["ocr_lang"],
    dpi=settings["ocr_dpi"],
)
extractor = HybridTextExtractor(
    native_extractor=NativeTextExtractor(),
    ocr_extractor=ocr_extractor,
    native_text_min_chars=settings["native_text_min_chars"],
)

# --- Scoring de pertinence (mots-clés + densité numérique + zero-shot optionnel) ---
scorer = build_default_composite_scorer(settings)

print("Extracteur:", extractor)
print("Scorer:", scorer)


## 5. Exécution du pipeline sur le dossier

In [ ]:
results = process_folder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    extractor=extractor,
    scorer=scorer,
    verbose=True,
)


## 6. Génération du rapport Excel global

In [ ]:
write_excel_report(results, EXCEL_REPORT_PATH)
print(f"PDF filtrés dans: {os.path.abspath(OUTPUT_DIR)}")
print(f"Rapport Excel: {os.path.abspath(EXCEL_REPORT_PATH)}")


## 7. Aperçu rapide des résultats dans le notebook

In [ ]:
from report import build_summary_dataframe, build_detail_dataframe

summary_df = build_summary_dataframe(results)
summary_df


In [ ]:
detail_df = build_detail_dataframe(results)
detail_df


## 8. (Optionnel) Tester la détection sur un seul texte

Pratique pour ajuster vos mots-clés dans `config.py` sans relancer tout le dossier.

In [ ]:
sample_text = """
Consolidated Statement of Financial Position
Total assets 1 234 567   Total liabilities 654 321
"""

evaluation = scorer.evaluate(sample_text)
print("Pertinent ?", evaluation.is_relevant)
print("Score:", evaluation.score)
print("Mots-clés trouvés:", evaluation.matched_keywords)
print("Détail:", evaluation.details)


---
### Notes sur la modularité

- **Changer d'OCR** : créez une nouvelle classe dans `text_extraction.py` héritant
  de `TextExtractor` (méthode `extract_page_texts`) et branchez-la dans la
  section 4. Rien d'autre à modifier.
- **Changer/ajouter des mots-clés** : éditez uniquement `config.py`
  (`KEYWORDS_EN`, `KEYWORDS_FR`).
- **Activer le zero-shot** : `pip install transformers torch`, puis
  `settings["use_zero_shot"] = True` dans la section 3. Le modèle par défaut
  (`joeddav/xlm-roberta-large-xnli`) est multilingue (EN/FR).
- **Changer la logique de fusion des scores** : modifiez les poids
  (`weight_keyword`, `weight_numeric_density`, `weight_zero_shot`) et le
  `relevance_threshold` dans `config.py`.
